# BART-large MNLI — DIMER zero-shot text classification tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/bart-mnli-zero-shot-classification-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/bart-mnli-zero-shot-classification-pipeline/blob/main/tutorials/bart_zero_shot_classification_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-facebook%2Fbart--large--mnli-ffcc4d?style=flat)](https://huggingface.co/facebook/bart-large-mnli) [![Upstream](https://img.shields.io/badge/Upstream-facebookresearch%2Ffairseq-181717?style=flat&logo=github&logoColor=white)](https://github.com/facebookresearch/fairseq/tree/main/examples/bart) [![arXiv](https://img.shields.io/badge/arXiv-1910.13461-b31b1b.svg)](https://arxiv.org/abs/1910.13461)

**Profile:** `TASK-INFERENCE`  
**Notebook specification:** DIMER Notebook Specification 1.1 — **standalone** (§3.6)  
**Capability:** zero-shot text classification by NLI entailment (one text, a caller-supplied label set, one entailment-derived score per label, sorted; single-label softmax across labels or independent multi-label scores) using the pinned BART-large MNLI weights

**This notebook is standalone.** It carries the repository's pipeline module (`src/bart_zero_shot_classification_pipeline/pipeline.py` at revision `cd035e9ec511`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned Python distributions and the Hugging Face Hub at the immutable revision `d7645e127eaf1aefc7862fd59a17a5aa8558b8ce` (~1632 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

At inference each caller-supplied label is inserted into the hypothesis template `This example is {label}.` (the upstream README's own recipe, exposed as `HYPOTHESIS_TEMPLATE`), every (text, hypothesis) pair is encoded as one sequence, and one batched forward pass of the 407 M-parameter BART-large encoder-decoder with its three-way MNLI classification head returns `contradiction`/`neutral`/`entailment` logits per pair. `classify` then converts them in NumPy: with `multi_label=False` the entailment logits are soft-maxed **across labels** (scores sum to one, `top_label` is the argmax); with `multi_label=True`, or when only one label is supplied, each label gets the entailment probability of a softmax over its own `[contradiction, entailment]` pair. **No adaptation occurs:** no training, fine-tuning, in-context conditioning, or preprocessing fitting — the pinned checkpoint is used as published. What the upstream checkpoint supplies is the encoder-decoder, the NLI head and the tokenizer; what the carried pipeline module adds is manifest verification, input validation and ceilings (over-long premise+hypothesis pairs are rejected, not truncated), the score conversion, a fixed output contract, the `accuracy` helper and the `validate_inputs` and `evaluation_report` stage helpers. **The `score` is entailment-derived and not a calibrated probability** of class membership; the pipeline ships no threshold.

**Learning objectives:** install the pinned runtime, read what the carried pipeline module guarantees, author three synthetic sentences with a three-label set and one author-expected label each (or upload your own), stage and digest-verify the immutable upstream snapshot, surface the pipeline's ceilings and validate the batch into one input manifest, run `classify` per sentence and read the ranked scores correctly in single-label and multi-label mode, read from the machine-readable evaluation report what the `accuracy` on three author-labelled sentences does and does not mean, and export scores alongside identifiers plus provenance.

**This notebook does not demonstrate:** supervised fine-tuning or a trained classifier, text generation or summarisation (the `bart-cnn-summarization-pipeline` sibling covers that), natural-language inference on caller-built premise/hypothesis pairs, sentence embeddings, token-level tagging, non-English text, or any calibrated probability. The repository exposes none of these.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU (float32) and uses CUDA automatically when available (also float32; the pipeline loads the checkpoint in float32 on both). This is a 407 M-parameter model: the model card's CPU smoke loaded and verified the 1.63 GB snapshot in 7.84 s and classified one sentence against three labels in 0.33 s, so the default path runs in well under a minute on a hosted CPU runtime once the ~1.6 GB `model.safetensors` download has finished. The pinned `torch==2.14.0` install and that download are the largest transfers of the run; allow ~2 GB of free RAM for the weights.
- **Knowledge:** basic Python; what natural-language inference (entailment vs contradiction) is; why a softmax over entailment logits is a ranking signal and not a calibrated probability.
- **Data:** the default sample is three synthetic sentences and a three-label set authored in code, each sentence with the label its author expects, so nothing is downloaded and no private data is needed. Optional BYOD upload is gated off by default so the sample path can run top-to-bottom without interaction. Expected BYOD input: one UTF-8 text file whose first non-empty line is the comma-separated label set and whose remaining non-empty lines are the texts to classify, each optionally followed by a tab and its gold label. Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded text remains in the notebook runtime; this pipeline does not send it to a third-party inference API.
- **External access:** the Hugging Face Hub only, to fetch the pinned `facebook/bart-large-mnli` snapshot (~1632 MB in total) at revision `d7645e127eaf…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same pins as the repository's pyproject.toml at the generating revision; any `--index-url`/`--find-links` lines are passed to pip as written) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'transformers==4.57.6',
    'tokenizers==0.22.2',
    'huggingface-hub==0.36.2',
    'safetensors==0.8.0',
    'numpy==2.5.3',
]
NOTEBOOK_SOURCE = {
    'repository': 'bart-mnli-zero-shot-classification-pipeline',
    'repository_revision': 'cd035e9ec511e70a68497d7a71d4615d6790b400',
    'embedded_module': 'src/bart_zero_shot_classification_pipeline/pipeline.py',
    'embedded_modules': ['src/bart_zero_shot_classification_pipeline/pipeline.py'],
    'module_sha256': 'b34bb1d2092eff3cd0507c02d1f8338bed6650c6b35090744637730d4d2b7fd6',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '1.1',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/bart_zero_shot_classification_pipeline/` @ `cd035e9ec511`)

The next 1 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/1:** `src/bart_zero_shot_classification_pipeline/pipeline.py`

In [ ]:
"""Zero-shot text classification by NLI entailment over the pinned ``facebook/bart-large-mnli``.

Weights load only from a digest-verified local snapshot (``weights/<key>/``) or, when explicitly allowed,
from the Hugging Face Hub at the pinned revision. One task method, ``classify``: the text is the NLI
premise, each caller-supplied label is turned into a hypothesis with ``HYPOTHESIS_TEMPLATE``, and the
entailment/contradiction logits are converted to one score per label (Yin et al., arXiv:1909.00161).
"""

from __future__ import annotations

import hashlib
import json
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np

MODEL_ID = "facebook/bart-large-mnli"
MODEL_REVISION = "d7645e127eaf1aefc7862fd59a17a5aa8558b8ce"
MODEL_LICENSE = "mit"
MODEL_KEY = "bart-large-mnli"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"

# Upstream hypothesis template (snapshot README "With manual PyTorch"): ``f'This example is {label}.'``.
HYPOTHESIS_TEMPLATE = "This example is {}."
# NLI head layout from the snapshot config.json ``label2id``: contradiction 0, neutral 1, entailment 2.
CONTRADICTION_INDEX = 0
ENTAILMENT_INDEX = 2
NUM_NLI_LABELS = 3
# Ceilings. 1024 is max_position_embeddings in config.json and model_max_length in tokenizer_config.json;
# a premise+hypothesis pair past it is rejected (not truncated) so a label is never scored on a cut premise.
MAX_TEXT_TOKENS = 1024
MAX_TEXT_CHARS = 8_000  # pre-tokenisation guard on the premise; ~4 chars per BPE token on English text
MAX_LABELS = 32  # hypotheses scored per classify() call (one forward pass, batched)
MAX_LABEL_CHARS = 100
DECISION_RULE_SINGLE = (
    "multi_label=False: softmax over the entailment logit across labels, argmax picks the label; "
    "no minimum score"
)
DECISION_RULE_MULTI = (
    "multi_label=True (or a single label): per label, softmax over [contradiction, entailment] logits, "
    "entailment probability is the score; no threshold applied"
)


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _read_manifest(root: Path) -> dict[str, Any]:
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"snapshot manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        return json.load(fh)


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its manifest; raise naming the first mismatch."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest = _read_manifest(root)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest.get("files", []):
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {"path": str(root), **manifest}


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest = _read_manifest(root)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def _check_text(text: Any, name: str = "text") -> str:
    if not isinstance(text, str):
        raise TypeError(f"{name} must be str, got {type(text).__name__}")
    if not text.strip():
        raise ValueError(f"{name} is empty")
    if len(text) > MAX_TEXT_CHARS:
        raise ValueError(f"{name} has {len(text)} chars; ceiling is MAX_TEXT_CHARS={MAX_TEXT_CHARS}")
    return text


def _check_labels(labels: Any) -> list[str]:
    """``classify``'s label contract; raise naming the first violated ceiling."""
    if isinstance(labels, str | bytes) or not isinstance(labels, Sequence):
        raise TypeError("labels must be a list of str, not a single string")
    if not 1 <= len(labels) <= MAX_LABELS:
        raise ValueError(f"labels must hold 1..MAX_LABELS={MAX_LABELS} items, got {len(labels)}")
    clean = []
    for i, label in enumerate(labels):
        if not isinstance(label, str):
            raise TypeError(f"labels[{i}] must be str, got {type(label).__name__}")
        if not label.strip():
            raise ValueError(f"labels[{i}] is empty")
        if len(label) > MAX_LABEL_CHARS:
            raise ValueError(f"labels[{i}] has {len(label)} chars; ceiling MAX_LABEL_CHARS={MAX_LABEL_CHARS}")
        clean.append(label)
    if len(set(clean)) != len(clean):
        raise ValueError("labels must be unique")
    return clean


def _check_template(template: Any) -> str:
    if not isinstance(template, str):
        raise TypeError("hypothesis_template must be str")
    if template.count("{}") != 1:
        raise ValueError("hypothesis_template must contain exactly one '{}' placeholder")
    return template


def _check_input_tokens(n_tokens: Sequence[int]) -> int:
    """The pair-token ceiling, applied once the tokenizer has counted every premise+hypothesis pair."""
    longest = max(int(n) for n in n_tokens)
    if longest > MAX_TEXT_TOKENS:
        raise ValueError(
            f"a premise+hypothesis pair is {longest} tokens; ceiling is MAX_TEXT_TOKENS={MAX_TEXT_TOKENS}"
        )
    return longest


INPUT_SCHEMA: dict[str, Any] = {
    "input": "one non-empty str (the NLI premise) and a list of unique non-empty str labels",
    "text_chars": [1, MAX_TEXT_CHARS],
    "pair_tokens": [1, MAX_TEXT_TOKENS],
    "labels": [1, MAX_LABELS],
    "label_chars": [1, MAX_LABEL_CHARS],
    "hypothesis_template": HYPOTHESIS_TEMPLATE,
    "multi_label": [False, True],
    "decision_rule": {"single": DECISION_RULE_SINGLE, "multi": DECISION_RULE_MULTI},
    "preprocessing": (
        "each label is inserted into hypothesis_template; every (premise, hypothesis) pair is BPE-encoded "
        "as one sequence with no truncation — a pair over MAX_TEXT_TOKENS is rejected, never cut"
    ),
}


def validate_inputs(
    texts: Sequence[str],
    labels: Sequence[str],
    *,
    multi_label: bool = False,
    hypothesis_template: str = HYPOTHESIS_TEMPLATE,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, per-input observations, verdict).

    ``classify`` takes one text per call, so ``texts`` is the batch the notebook will loop over; every
    entry and the shared ``labels``/``hypothesis_template`` go through the same private checks the
    method uses (``_check_text``, ``_check_labels``, ``_check_template``), so a rejection here is a
    rejection there. ``MAX_TEXT_TOKENS`` needs the loaded tokenizer and is enforced inside ``classify``.
    """
    if isinstance(texts, str | bytes) or not isinstance(texts, Sequence):
        raise TypeError("texts must be a sequence of str, not a single string")
    if not texts:
        raise ValueError("texts must hold at least one item")
    checked = [_check_text(text, f"texts[{i}]") for i, text in enumerate(texts)]
    clean_labels = _check_labels(labels)
    template = _check_template(hypothesis_template)
    if not isinstance(multi_label, bool):
        raise TypeError("multi_label must be a bool")
    if names is not None and len(names) != len(checked):
        raise ValueError("names must have one entry per text")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [
            {"id": names[i] if names else f"text-{i}", "chars": len(text)} for i, text in enumerate(checked)
        ],
        "labels": clean_labels,
        "hypotheses": [template.format(label) for label in clean_labels],
        "multi_label": multi_label,
        "hypothesis_template": template,
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def accuracy(predicted: Sequence[str], gold: Sequence[str]) -> float:
    """Fraction of items whose top label equals the gold label (exact string match)."""
    if len(predicted) != len(gold):
        raise ValueError("predicted and gold must have the same length")
    if not gold:
        raise ValueError("accuracy needs at least one item")
    return float(sum(p == g for p, g in zip(predicted, gold, strict=True)) / len(gold))


def evaluation_report(
    results: Sequence[Mapping[str, Any]],
    gold_labels: Sequence[str] | None = None,
    *,
    sample_kind: str = "synthetic",
) -> dict[str, Any]:
    """Evaluation stage: ``accuracy`` of the top label against caller-supplied gold labels, else
    ``not-measurable``. The number is a sample-sanity observation on however many items were passed,
    never a benchmark; the score behind it is entailment-derived and not calibrated."""
    predicted = [str(result.get("top_label")) for result in results]
    supplied = gold_labels is not None
    report: dict[str, Any] = {
        "task": "zero-shot text classification by NLI entailment (caller-supplied label set)",
        "score_semantics": (
            "score is an entailment-derived softmax — over labels when multi_label=False, over "
            "[contradiction, entailment] per label otherwise — a ranking signal, not a calibrated "
            "probability; the decision rule is argmax over score and no threshold is shipped"
        ),
        "sample_kind": sample_kind,
        "n_items": len(predicted),
        "metrics": [],
        "baselines": [],
        "verdict": "not-measurable",
        "reason": "no gold labels were supplied, so the top labels cannot be scored",
        "needs": (
            "one gold label per text from the deployment's own label set over enough texts to state a "
            "dispersion; the `accuracy` helper then scores exact top-label matches, and a calibration set "
            "is needed before any score is read as a probability"
        ),
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }
    if supplied:
        value = accuracy(predicted, list(gold_labels))
        report["metrics"] = [
            {
                "id": "accuracy",
                "value": value,
                "estimation": f"single pass over {len(predicted)} item(s); no dispersion",
                "decision_rule": "top_label == gold label (exact string match)",
            }
        ]
        report["verdict"] = "sample-sanity"
        report["reason"] = (
            f"gold labels were supplied for {len(predicted)} item(s); the accuracy is a plumbing check on "
            "that sample, not a benchmark"
        )
    return report


@dataclass
class BARTZeroShotClassificationPipeline:
    """``_runner(text, hypotheses)`` -> (NLI logits ``(n_hypotheses, 3)``, per-pair token counts). Injectable
    so tests run offline."""

    _runner: Callable[[str, list[str]], tuple[np.ndarray, list[int]]]
    device: str = "cpu"
    source: str = "injected"

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> BARTZeroShotClassificationPipeline:
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            location, kwargs, source = str(root), {"local_files_only": True}, "local-snapshot"
        elif allow_download:
            location, kwargs, source = MODEL_ID, {}, "hf-hub"
        else:
            raise FileNotFoundError(
                f"no verified snapshot at {root} and allow_download=False; "
                f"stage {MODEL_ID}@{MODEL_REVISION} under weights/{MODEL_KEY}"
            )
        # Refuse invalid snapshots before importing model libraries.
        import torch
        from transformers import AutoTokenizer, BartForSequenceClassification

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        tokenizer = AutoTokenizer.from_pretrained(
            location, revision=MODEL_REVISION, trust_remote_code=False, **kwargs
        )
        model = BartForSequenceClassification.from_pretrained(
            location, revision=MODEL_REVISION, dtype=torch.float32, trust_remote_code=False, **kwargs
        )
        model = model.to(resolved_device).eval()

        def runner(text: str, hypotheses: list[str]) -> tuple[np.ndarray, list[int]]:
            batch = tokenizer(
                [text] * len(hypotheses), hypotheses, return_tensors="pt", padding=True, truncation=False
            )
            n_tokens = [int(n) for n in batch["attention_mask"].sum(dim=1)]
            _check_input_tokens(n_tokens)
            with torch.inference_mode():
                logits = model(**batch.to(resolved_device)).logits
            return logits.float().cpu().numpy(), n_tokens

        return cls(runner, resolved_device, source)

    def classify(
        self,
        text: str,
        labels: Sequence[str],
        *,
        multi_label: bool = False,
        hypothesis_template: str = HYPOTHESIS_TEMPLATE,
    ) -> dict[str, Any]:
        """Score every label against ``text``; ``labels`` in the result are sorted by descending score."""
        text = _check_text(text)
        clean = _check_labels(labels)
        template = _check_template(hypothesis_template)
        if not isinstance(multi_label, bool):
            raise TypeError("multi_label must be a bool")
        hypotheses = [template.format(label) for label in clean]
        logits, n_tokens = self._runner(text, hypotheses)
        logits = np.asarray(logits, dtype=np.float64)
        if logits.shape != (len(clean), NUM_NLI_LABELS):
            raise RuntimeError(f"backend returned {logits.shape}, expected ({len(clean)}, {NUM_NLI_LABELS})")
        longest = _check_input_tokens(n_tokens)
        # Upstream ZeroShotClassificationPipeline rule: a single label always takes the per-label path.
        per_label = multi_label or len(clean) == 1
        if per_label:
            pair = logits[:, [CONTRADICTION_INDEX, ENTAILMENT_INDEX]]
            shifted = np.exp(pair - pair.max(axis=1, keepdims=True))
            scores = (shifted / shifted.sum(axis=1, keepdims=True))[:, 1]
        else:
            entail = logits[:, ENTAILMENT_INDEX]
            shifted = np.exp(entail - entail.max())
            scores = shifted / shifted.sum()
        order = np.argsort(-scores, kind="stable")
        ranked = [
            {
                "label": clean[i],
                "score": float(scores[i]),
                "entailment_logit": float(logits[i, ENTAILMENT_INDEX]),
                "contradiction_logit": float(logits[i, CONTRADICTION_INDEX]),
            }
            for i in order
        ]
        return {
            "labels": ranked,
            "top_label": ranked[0]["label"],
            "multi_label": multi_label,
            "hypothesis_template": template,
            "decision_rule": DECISION_RULE_MULTI if per_label else DECISION_RULE_SINGLE,
            "n_tokens": longest,
            "device": self.device,
            "source": self.source,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `7`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `d7645e127eaf…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `BARTZeroShotClassificationPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "bart-large-mnli",
  "modelId": "facebook/bart-large-mnli",
  "revision": "d7645e127eaf1aefc7862fd59a17a5aa8558b8ce",
  "files": [
    {
      "path": "README.md",
      "bytes": 3793,
      "sha256": "d022b7cb54ca2f65fac41cf6c9601b4491b4ac22a301227288a38beef2a18aeb"
    },
    {
      "path": "config.json",
      "bytes": 1154,
      "sha256": "a0f9bcb245b680a96ccae0ad8d155f267ec3e3c971ef4a4937e52ea9ba368a86"
    },
    {
      "path": "merges.txt",
      "bytes": 456318,
      "sha256": "1ce1664773c50f3e0cc8842619a93edc4624525b728b188a9e0be33b7726adc5"
    },
    {
      "path": "model.safetensors",
      "bytes": 1629437147,
      "sha256": "cfbb687dbbd9df99fe865e1860350a22aebac4d26ee4bcb50217f1df606a018e"
    },
    {
      "path": "tokenizer.json",
      "bytes": 1355863,
      "sha256": "847bbeab6174d66a88898f729d52fa8d355fafe1bea101cf960dd404581df70e"
    },
    {
      "path": "tokenizer_config.json",
      "bytes": 26,
      "sha256": "5e04eb606e3a1583530a42e36c2a6b6615c86f34fe77e44d9ddeb43ff940931f"
    },
    {
      "path": "vocab.json",
      "bytes": 898822,
      "sha256": "06b4d46c8e752d410213d9548eb27a54db70fda0319b6271fb8d59dead5e1cab"
    }
  ],
  "totalBytes": 1632153123
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = BARTZeroShotClassificationPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Author the synthetic sample or optional BYOD

The default sample is **synthetic**, written in this cell: three sentences — the upstream README's own example (`one day I will see the world`) and two more — against the label set `travel`, `cooking`, `dancing`, each sentence carrying the label its author expects and a stable identifier (`t1`, `t2`, `t3`) so every score can be mapped back to its text. The expected labels are the author's intent, not a labelled dataset: the `accuracy` the evaluation stage computes on them is a sample-sanity check that the code path works, never benchmark evidence. `MULTI_LABEL` is a Colab form parameter checked against the carried module in Section 5.

BYOD is optional and disabled by default. Expected BYOD input: one UTF-8 text file whose first non-empty line is the comma-separated label set (1..`MAX_LABELS` unique labels, each at most `MAX_LABEL_CHARS` characters) and whose remaining non-empty lines are the texts to classify (each at most `MAX_TEXT_CHARS` characters and, paired with the longest hypothesis, at most `MAX_TEXT_TOKENS` BPE tokens — longer pairs are rejected by the pipeline, not truncated), each optionally followed by a tab and its gold label. If any line carries a gold label, every line must. The upload stays inside this runtime.

In [ ]:
import hashlib
import io

USE_BYOD = False  # @param {type:"boolean"}
MULTI_LABEL = False  # @param {type:"boolean"}

if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    sample_name = next(iter(uploaded))
    lines = [line.rstrip('\n') for line in io.StringIO(uploaded[sample_name].decode('utf-8')) if line.strip()]
    if len(lines) < 2:
        raise ValueError(f'{sample_name}: expected a label line followed by at least one text line')
    labels = [label.strip() for label in lines[0].split(',') if label.strip()]
    rows = [line.split('\t', 1) for line in lines[1:]]
    texts = [row[0].strip() for row in rows]
    golds = [row[1].strip() for row in rows if len(row) == 2]
    if golds and len(golds) != len(texts):
        raise ValueError(f'{sample_name}: {len(golds)} of {len(texts)} lines carry a gold label; all or none must')
    gold_labels = golds or None
    sample_kind = 'BYOD upload'
else:
    labels = ['travel', 'cooking', 'dancing']
    texts = [
        'one day I will see the world',
        'Whisk the eggs and fold in the flour before baking.',
        'The tango class meets every Thursday evening.',
    ]
    gold_labels = ['travel', 'cooking', 'dancing']
    sample_name = 'synthetic_three_sentences'
    sample_kind = 'synthetic (authored in this cell; the first sentence is the upstream README example)'
text_ids = [f't{index + 1}' for index in range(len(texts))]
sample_sha256 = hashlib.sha256('\n'.join([','.join(labels), *texts]).encode('utf-8')).hexdigest()
print({'sample': sample_name, 'sample_kind': sample_kind, 'labels': labels, 'texts': len(texts), 'gold_labels': gold_labels, 'multi_label': MULTI_LABEL, 'text_sha256': sample_sha256})
for text_id, text in zip(text_ids, texts, strict=True):
    print(f'{text_id}: {text[:100]}')

## 5. Validate the inputs → input manifest

`validate_inputs` is the pipeline's public validation stage: it takes the batch of texts `classify` will be called on one by one, the shared label set, `multi_label` and the hypothesis template, and runs the method's own checks — `_check_text`, `_check_labels`, `_check_template` — so a rejection here is a rejection there. `MAX_TEXT_CHARS` is the character guard applied before tokenisation; `MAX_TEXT_TOKENS` (1024, the checkpoint's position limit) applies to each premise+hypothesis pair after tokenisation and **rejects** longer pairs rather than truncating them, so it is enforced inside the pipeline and cannot be observed at this stage; `MAX_LABELS` and `MAX_LABEL_CHARS` bound the label set; `HYPOTHESIS_TEMPLATE` is the sentence each label is inserted into, and the manifest lists the resulting `hypotheses` verbatim so you can read whether your labels make natural sentences. The manifest is written to `outputs/bart_zero_shot_classification_input_manifest.json`. To show what rejection looks like, the cell also validates a label set with a duplicate entry and records the pipeline's own error message as a finding. The notebook never trims or alters the texts or labels.

In [ ]:
import json

os.makedirs('outputs', exist_ok=True)
ceilings = {'MAX_TEXT_CHARS': MAX_TEXT_CHARS, 'MAX_TEXT_TOKENS': MAX_TEXT_TOKENS, 'MAX_LABELS': MAX_LABELS, 'MAX_LABEL_CHARS': MAX_LABEL_CHARS, 'HYPOTHESIS_TEMPLATE': HYPOTHESIS_TEMPLATE, 'NUM_NLI_LABELS': NUM_NLI_LABELS, 'ENTAILMENT_INDEX': ENTAILMENT_INDEX, 'CONTRADICTION_INDEX': CONTRADICTION_INDEX}
print(ceilings)
input_manifest = validate_inputs(texts, labels, multi_label=MULTI_LABEL, names=text_ids)
# Demonstrate the unique-labels rejection; the finding is recorded, not swallowed.
try:
    validate_inputs(texts, [*labels, labels[0]], multi_label=MULTI_LABEL)
except ValueError as exc:
    input_manifest['findings'].append({'input': 'duplicate-label-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/bart_zero_shot_classification_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
print(json.dumps(input_manifest, indent=2))
print({'token_ceiling': f'MAX_TEXT_TOKENS={MAX_TEXT_TOKENS} is checked per premise+hypothesis pair by the pipeline after tokenisation and rejects, never truncates'})

## 6. Classify each sentence and read the scores correctly

**Input/output contract.** `classify(text, labels, multi_label=...)` takes one string and the label list and returns `labels` — one entry per supplied label ordered by descending `score`, each with the `label`, the `score`, and the raw `entailment_logit` and `contradiction_logit` behind it — plus `top_label`, `multi_label`, the `hypothesis_template` used, the `decision_rule` applied, `n_tokens` (the longest premise+hypothesis pair), the device and the model identity. **Score semantics:** with `multi_label=False` the score is a softmax over the entailment logits **across the labels** — it sums to one over the label set and the **default decision rule is `argmax`** (the first entry), so a text that fits none of the labels still gets a winner; with `multi_label=True` each score is the entailment probability of a softmax over that label's own `[contradiction, entailment]` pair, the scores do not sum to one, and no threshold is applied. In neither mode is the score a calibrated probability of class membership: it moves with the label wording and the template, and any acceptance threshold is owned by the caller and must be set on their own labelled data. The model card's CPU smoke on the first sentence gave `travel` 0.9939, `dancing` 0.0033, `cooking` 0.0029 — the upstream README's own numbers to four decimals — which is one observation, not an expected value; near-tied labels can reorder between CPU and CUDA kernels. Inference is deterministic on a fixed device and dtype (`model.eval()`, no sampling, no seed needed). The cell also classifies the first sentence in the other mode so the two score conventions can be compared side by side.

In [ ]:
import time

results = []
elapsed = []
for text_id, text in zip(text_ids, texts, strict=True):
    started = time.perf_counter()
    result = pipe.classify(text, labels, multi_label=MULTI_LABEL)
    elapsed.append(time.perf_counter() - started)
    results.append(result)
    scores = [entry['score'] for entry in result['labels']]
    checks = {
        'one_entry_per_label': sorted(entry['label'] for entry in result['labels']) == sorted(labels),
        'scores_descending': all(a >= b for a, b in zip(scores, scores[1:], strict=False)),
        'scores_in_unit_interval': all(0.0 <= s <= 1.0 for s in scores),
        'single_label_scores_sum_to_one': MULTI_LABEL or len(labels) == 1 or abs(sum(scores) - 1.0) < 1e-6,
        'top_label_is_first': result['top_label'] == result['labels'][0]['label'],
        'n_tokens_within_ceiling': 1 <= result['n_tokens'] <= MAX_TEXT_TOKENS,
    }
    if not all(checks.values()):
        raise RuntimeError(f'classify output for {text_id} failed a sanity check: {checks}')
    print(f'{text_id}: {text[:60]}')
    print({'top_label': result['top_label'], 'seconds': round(elapsed[-1], 3), 'n_tokens': result['n_tokens'], 'decision_rule': result['decision_rule'], 'checks': checks})
    for rank, entry in enumerate(result['labels'], start=1):
        print(f"  {rank}. {entry['label']:<12} score {entry['score']:.4f}  entailment {entry['entailment_logit']:+.3f}  contradiction {entry['contradiction_logit']:+.3f}")
other_mode = pipe.classify(texts[0], labels, multi_label=not MULTI_LABEL)
print({'first_text_other_mode': {'multi_label': other_mode['multi_label'], 'scores': {entry['label']: round(entry['score'], 4) for entry in other_mode['labels']}, 'decision_rule': other_mode['decision_rule']}})
classify_sanity = {}
if gold_labels is not None:
    classify_sanity = {'top_label_matches_expected': [result['top_label'] == gold for result, gold in zip(results, gold_labels, strict=True)]}
    print({'sanity_check': classify_sanity, 'note': 'falsifiable plumbing check on author-expected labels; the metric is computed in Section 7'})

## 7. Evaluate → evaluation report

`evaluation_report` is the pipeline's public evaluation stage and always produces a report. When gold labels are supplied it reports one metric, **`accuracy`** — the fraction of texts whose `top_label` equals the gold label by exact string match, the repository's own `accuracy` helper — with verdict `sample-sanity`: on three author-labelled synthetic sentences the number is a plumbing check on this sample, estimated by a single pass with no dispersion, and the report says so in `estimation` and `reason`. With `multi_label=True` the argmax match is only a proxy, because the mode exists for texts that belong to several classes. Without gold labels the verdict is `not-measurable` and `needs` names what would make the task measurable: one gold label per text from the deployment's own label set over enough texts to state a dispersion, and a calibration set before any score is read as a probability. The sanity checks printed in Section 6 remain falsifiable plumbing checks, not results. The report is written to `outputs/bart_zero_shot_classification_evaluation_report.json`.

In [ ]:
report = evaluation_report(results, gold_labels, sample_kind=sample_kind)
with open('outputs/bart_zero_shot_classification_evaluation_report.json', 'w', encoding='utf-8') as handle:
    json.dump(report, handle, indent=2, ensure_ascii=False)
print(json.dumps(report, indent=2))
if report['verdict'] == 'not-measurable':
    print('No metric is reported: supply one gold label per text to obtain the accuracy plumbing check; a real evaluation needs your own labelled set.')
else:
    print({'accuracy': report['metrics'][0]['value'], 'note': 'sample-sanity on ' + str(report['n_items']) + ' author-labelled item(s); not a benchmark'})

## 8. Export scores alongside identifiers, and provenance

Two further files are written under `outputs/` beside the input manifest and the evaluation report. The scores go to CSV (`outputs/bart_zero_shot_classification_scores.csv`) with one row per (text, label) — `text_id`, `rank`, `label`, `score`, `entailment_logit`, `contradiction_logit`, `multi_label` — so every score stays attached to its text identifier for downstream use. One JSON record (`outputs/bart_zero_shot_classification_result.json`) preserves every result (the identified texts, the ranked labels with scores and logits, `top_label`, the decision rule, `n_tokens`, seconds), the other-mode comparison, the sanity checks, the ceilings in force, the input manifest, the evaluation report, the sample identity and digest, the notebook's source (repository, revision, embedded module digest, generator), the model identifier, the immutable model revision, the model licence, the verified snapshot summary, and the runtime identity (Python, `torch`, `transformers`, device, dtype). No credentials are involved in any step, so none can reach the export.

In [ ]:
import csv

with open('outputs/bart_zero_shot_classification_scores.csv', 'w', encoding='utf-8', newline='') as handle:
    writer = csv.writer(handle)
    writer.writerow(['text_id', 'rank', 'label', 'score', 'entailment_logit', 'contradiction_logit', 'multi_label'])
    for text_id, result in zip(text_ids, results, strict=True):
        for rank, entry in enumerate(result['labels'], start=1):
            writer.writerow([text_id, rank, entry['label'], f"{entry['score']:.7f}", f"{entry['entailment_logit']:.5f}", f"{entry['contradiction_logit']:.5f}", result['multi_label']])
payload = {
    'classify': [
        {
            'text_id': text_id,
            'text': text,
            'expected_label': None if gold_labels is None else gold,
            'top_label': result['top_label'],
            'labels': [{'rank': rank, **entry} for rank, entry in enumerate(result['labels'], start=1)],
            'decision_rule': result['decision_rule'],
            'multi_label': result['multi_label'],
            'hypothesis_template': result['hypothesis_template'],
            'n_tokens': result['n_tokens'],
            'seconds': round(seconds, 3),
        }
        for text_id, text, gold, result, seconds in zip(text_ids, texts, gold_labels or [None] * len(texts), results, elapsed, strict=True)
    ],
    'score_semantics': 'entailment-derived softmax (across labels when multi_label=False, per label otherwise); not a calibrated probability; argmax rule; no threshold shipped',
    'other_mode_first_text': {'multi_label': other_mode['multi_label'], 'labels': other_mode['labels'], 'decision_rule': other_mode['decision_rule']},
    'plumbing_check': classify_sanity,
    'scores_file': 'outputs/bart_zero_shot_classification_scores.csv',
    'ceilings': ceilings,
    'input_manifest': input_manifest,
    'evaluation_report': report,
    'sample': {'name': sample_name, 'kind': sample_kind, 'labels': labels, 'text_sha256': sample_sha256},
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'snapshot': {'path': snapshot['path'], 'files': len(snapshot['files']), 'total_bytes': snapshot.get('totalBytes')},
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'transformers': transformers.__version__,
        'device': pipe.device,
        'dtype': 'float32',
    },
}
with open('outputs/bart_zero_shot_classification_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False)
print(sorted(os.listdir('outputs')))

## Interpretation and limits

The scores are entailment-derived: each label became the hypothesis `This example is {label}.`, the MNLI head judged it against the text, and the entailment logits were soft-maxed across the labels (single-label mode) or against each label's own contradiction logit (multi-label mode). In single-label mode the first entry is the argmax and the scores sum to one, so a text that matches none of the labels still receives a confident-looking winner; in multi-label mode nothing is thresholded. Neither score is a calibrated probability, both move with the label wording and the template, and the pipeline applies no threshold — the caller owns any cut-off and must set it on labelled data from their own label set. On the synthetic sample the `accuracy` in the evaluation report is a sample-sanity check over three author-labelled sentences with no dispersion, and a real evaluation needs the caller's own labelled texts over enough items to state one. Premise+hypothesis pairs over 1024 BPE tokens are rejected, not truncated; the checkpoint is English only and carries whatever associations MultiNLI and the BART pre-training corpus contain, which neither the upstream card nor this repository has audited; the pipeline exposes no fine-tuning, no generation and no raw NLI on caller-built hypotheses. Inference is deterministic on a fixed device and dtype, but CPU and CUDA kernels can reorder near-tied labels.

Successful execution proves that the recorded repository revision's pipeline module, carried in this notebook, can acquire and digest-verify the pinned model snapshot, validate the demonstrated inputs against the enforced ceilings, execute the public `classify` path in both modes, and emit the shown machine-readable outputs in the tested runtime — without the repository being reachable. It does **not** establish benchmark superiority, classification accuracy on any domain, a usable threshold, safety for high-consequence decisions, or production fitness on an unseen domain.

**Next experiments.** Flip `MULTI_LABEL` and compare the two score conventions on the same sentences; reword one label (`travel` → `holidays`) and watch every score move, which is what "not calibrated" means in practice; add a sentence that fits none of the labels and read the single-label winner it still receives; assemble a few dozen labelled texts of your own and compute the accuracy the evaluation report asks for, with a bootstrap interval. None of these turns the sample result into evidence of production fitness.

## References

- Repository README: https://github.com/kurtvalcorza/bart-mnli-zero-shot-classification-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/bart-mnli-zero-shot-classification-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/bart-mnli-zero-shot-classification-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model: https://huggingface.co/facebook/bart-large-mnli
- Upstream code: https://github.com/facebookresearch/fairseq/tree/main/examples/bart
- BART: Denoising Sequence-to-Sequence Pre-training for Natural Language Generation, Translation, and Comprehension: https://arxiv.org/abs/1910.13461
- Benchmarking Zero-shot Text Classification: Datasets, Evaluation and Entailment Approach (Yin et al.): https://arxiv.org/abs/1909.00161